# Q1 2026 AI-chip component-spend nowcast

Visualizes `pipelines/nowcast` output: the 9-quarter history (4 designers) plus the
Q1 2026 nowcast band, and how each estimate family shifts the number. See the module
README for method, sources, and limitations. Run `uv run -m pipelines.nowcast all` first.

In [1]:
import polars as pl
import plotly.express as px
import plotly.graph_objects as go

D = "../data/processed/nowcast"
hist = pl.read_csv(f"{D}/nowcast_history.csv")
now = pl.read_csv(f"{D}/nowcast_q1_2026.csv")
detail = pl.read_csv(f"{D}/nowcast_estimates_detail.csv")
comps = ["Memory", "Logic", "Packaging", "Auxiliary"]

def qkey(q):
    n, y = q.split(" ")
    return int(y) * 4 + int(n[1])

hist = hist.with_columns(pl.col("Quarter").map_elements(qkey, return_dtype=pl.Int64).alias("_k")).sort("_k")
complete = hist.filter(pl.col("Quarter") != "Q1 2026")
tot = now.filter(pl.col("component") == "TOTAL").row(0, named=True)

## Total component spend: history + Q1 2026 nowcast band

In [2]:
hist_tot = complete.with_columns(total=sum(pl.col(c) for c in comps))
fig = go.Figure()
fig.add_scatter(x=hist_tot["Quarter"].to_list(), y=hist_tot["total"].to_list(),
                mode="lines+markers", name="history (actual)")
fig.add_scatter(x=["Q1 2026"], y=[tot["reconciled_base"]], mode="markers", name="Q1 2026 nowcast",
                marker=dict(size=11, color="crimson"),
                error_y=dict(type="data", symmetric=False,
                             array=[tot["high"] - tot["reconciled_base"]],
                             arrayminus=[tot["reconciled_base"] - tot["low"]]))
fig.add_scatter(x=["Q1 2026"], y=[tot["trend_only"]], mode="markers", name="trend-only",
                marker=dict(size=9, color="gray", symbol="x"))
fig.update_layout(title="AI-chip component spend (NVIDIA/AMD/Google/Amazon): history + Q1 2026 nowcast",
                  yaxis_title="USD billions", height=500)
fig

## Per-component nowcast (reconciled base with low–high band)

In [3]:
pc = now.filter(pl.col("component") != "TOTAL")
fig = go.Figure(go.Bar(
    x=pc["component"].to_list(), y=pc["reconciled_base"].to_list(),
    error_y=dict(type="data", symmetric=False,
                 array=[h - b for h, b in zip(pc["high"], pc["reconciled_base"])],
                 arrayminus=[b - l for b, l in zip(pc["reconciled_base"], pc["low"])]),
    marker_color=["#4fa8a0", "#e0a44a", "#4a5fd0", "#d65a9a"],
    text=[f"{v:.1f}" for v in pc["reconciled_base"]]))
fig.update_layout(title="Q1 2026 reconciled nowcast by component (low–high band)",
                  yaxis_title="USD billions", height=450)
fig

## Per-source contribution to each estimate
How each family (trend / supply / demand / price / macro) lands per component, and the
reconciled blend. Divergence between families is the signal to read.

In [4]:
order = ["trend", "supply", "demand", "price", "macro", "reconciled"]
fig = px.bar(detail.to_pandas(), x="estimate", y="value_b", facet_col="component",
             color="estimate", category_orders={"estimate": order, "component": comps},
             labels={"value_b": "USD billions"},
             title="Estimate by family, per component (base scenario)")
fig.update_yaxes(matches=None)
fig.update_layout(height=450, showlegend=False)
fig